# 📊 Day 03a: Pandas GroupBy & Aggregations — Think in SQL, Write in Pandas

---

## 🎯 What You'll Master Today
- groupby() mechanics
- agg() with multiple functions
- transform() vs apply()
- pivot_table() and crosstab()
- Multi-level groupby

**Goal:** Replicate any SQL GROUP BY in Pandas from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  GroupBy — Split-Apply-Combine                                   ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. SPLIT:    Group rows by key column(s)                       ║
║  2. APPLY:    Apply function to each group                      ║
║  3. COMBINE:  Merge results back together                       ║
║                                                                   ║
║  .agg()       → reduce each group to one row (like SQL)        ║
║  .transform() → return same-size result (broadcast back)       ║
║  .apply()     → flexible, any function                          ║
║                                                                   ║
║  SQL equiv:                                                       ║
║    SELECT dept, AVG(salary) FROM emp GROUP BY dept             ║
║    → df.groupby('dept')['salary'].mean()                       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import pandas as pd
import numpy as np

# Sample dataset
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Hank'],
    'dept': ['Eng', 'Sales', 'Eng', 'HR', 'Sales', 'Eng', 'HR', 'Sales'],
    'level': ['Senior', 'Junior', 'Senior', 'Junior', 'Senior', 'Junior', 'Senior', 'Junior'],
    'salary': [90000, 50000, 85000, 45000, 70000, 65000, 55000, 48000],
    'bonus': [10000, 5000, 9000, 4000, 7000, 6000, 5500, 4500],
    'sales': [None, 120, None, None, 150, None, None, 90]
})
print(df)

In [ ]:
# ============================================================
# 1. Basic GroupBy
# ============================================================

# Single column group, single aggregation
# SQL: SELECT dept, AVG(salary) FROM emp GROUP BY dept
print(f"Avg salary by dept:\n{df.groupby('dept')['salary'].mean()}\n")

# Multiple aggregations
print(f"Stats by dept:\n{df.groupby('dept')['salary'].agg(['mean', 'min', 'max', 'count'])}\n")

# Group by multiple columns
# SQL: SELECT dept, level, AVG(salary) FROM emp GROUP BY dept, level
print(f"By dept + level:\n{df.groupby(['dept', 'level'])['salary'].mean()}")

In [ ]:
# ============================================================
# 2. .agg() — Multiple Functions, Multiple Columns
# ============================================================

# Same function on multiple columns
result = df.groupby('dept')[['salary', 'bonus']].mean()
print(f"Mean salary + bonus:\n{result}\n")

# Different functions per column
result = df.groupby('dept').agg({
    'salary': ['mean', 'max'],
    'bonus': 'sum',
    'name': 'count'
})
print(f"Custom agg:\n{result}\n")

# Named aggregations (clean column names)
result = df.groupby('dept').agg(
    avg_salary=('salary', 'mean'),
    max_salary=('salary', 'max'),
    total_bonus=('bonus', 'sum'),
    headcount=('name', 'count')
)
print(f"Named agg (clean!):\n{result}")
print("\n💡 Named agg = best practice for readable output")

In [ ]:
# ============================================================
# 3. .transform() — Broadcast Result Back to Original Size
# ============================================================

# KEY DIFFERENCE:
# agg()      → reduces (one row per group)
# transform() → same size as input (broadcasts back)

# Add column: department average salary
df['dept_avg'] = df.groupby('dept')['salary'].transform('mean')
print(f"With dept avg:\n{df[['name', 'dept', 'salary', 'dept_avg']]}\n")

# Deviation from department mean
df['salary_vs_dept'] = df['salary'] - df.groupby('dept')['salary'].transform('mean')
print(f"Salary vs dept avg:\n{df[['name', 'dept', 'salary', 'salary_vs_dept']]}\n")

# Normalize within group (z-score per department)
df['salary_zscore'] = df.groupby('dept')['salary'].transform(
    lambda x: (x - x.mean()) / x.std()
)
print(f"Z-score within dept:\n{df[['name', 'dept', 'salary', 'salary_zscore']].round(2)}")

# Clean up
df = df.drop(columns=['dept_avg', 'salary_vs_dept', 'salary_zscore'])

In [ ]:
# ============================================================
# 4. .apply() — Full Flexibility
# ============================================================

# Apply a custom function to each group
def top_earner(group):
    return group.nlargest(1, 'salary')

result = df.groupby('dept').apply(top_earner, include_groups=False)
print(f"Top earner per dept:\n{result}\n")

# Return a summary Series per group
def group_summary(group):
    return pd.Series({
        'count': len(group),
        'avg_salary': group['salary'].mean(),
        'salary_range': group['salary'].max() - group['salary'].min()
    })

result = df.groupby('dept').apply(group_summary, include_groups=False)
print(f"Group summaries:\n{result}")

In [ ]:
# ============================================================
# 5. pivot_table() — Spreadsheet-style Summarization
# ============================================================

# Basic pivot table
# SQL: SELECT dept, level, AVG(salary) FROM emp GROUP BY dept, level
pivot = pd.pivot_table(
    df, 
    values='salary', 
    index='dept', 
    columns='level', 
    aggfunc='mean'
)
print(f"Pivot (avg salary):\n{pivot}\n")

# Multiple aggregations
pivot2 = pd.pivot_table(
    df,
    values='salary',
    index='dept',
    columns='level',
    aggfunc=['mean', 'count'],
    fill_value=0
)
print(f"Pivot (mean + count):\n{pivot2}\n")

# With margins (totals)
pivot3 = pd.pivot_table(
    df,
    values='salary',
    index='dept',
    columns='level',
    aggfunc='mean',
    margins=True,
    margins_name='Total'
)
print(f"Pivot with totals:\n{pivot3}")

In [ ]:
# ============================================================
# 6. crosstab() — Frequency Tables
# ============================================================

# Count of employees per dept x level
ct = pd.crosstab(df['dept'], df['level'])
print(f"Crosstab (count):\n{ct}\n")

# With margins and normalize
ct2 = pd.crosstab(df['dept'], df['level'], margins=True, normalize='index')
print(f"Crosstab (% by row):\n{ct2.round(2)}\n")

print("💡 crosstab = quick frequency table, pivot_table = flexible aggregation")

In [ ]:
# ============================================================
# 7. Real-World GroupBy Patterns
# ============================================================

# Pattern 1: Filter groups (HAVING in SQL)
# SELECT dept FROM emp GROUP BY dept HAVING COUNT(*) >= 3
big_depts = df.groupby('dept').filter(lambda g: len(g) >= 3)
print(f"Depts with 3+ people:\n{big_depts[['name', 'dept']]}\n")

# Pattern 2: Rank within group
df['salary_rank'] = df.groupby('dept')['salary'].rank(ascending=False).astype(int)
print(f"Salary rank per dept:\n{df[['name', 'dept', 'salary', 'salary_rank']]}\n")

# Pattern 3: Cumulative sum within group
df['cum_bonus'] = df.groupby('dept')['bonus'].cumsum()
print(f"Cumulative bonus:\n{df[['name', 'dept', 'bonus', 'cum_bonus']]}\n")

# Pattern 4: Percentage of group total
df['pct_of_dept'] = df['salary'] / df.groupby('dept')['salary'].transform('sum') * 100
print(f"% of dept salary:\n{df[['name', 'dept', 'salary', 'pct_of_dept']].round(1)}")

# Clean up
df = df.drop(columns=['salary_rank', 'cum_bonus', 'pct_of_dept'])

---

## 🗺️ GroupBy Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  GroupBy — SQL to Pandas Translation                             ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  GROUP BY + AGG    → .groupby('col').agg(...)                   ║
║  HAVING            → .groupby().filter(lambda g: ...)           ║
║  Window func       → .groupby().transform(func)                ║
║  RANK()            → .groupby()['col'].rank()                  ║
║  SUM() OVER        → .groupby()['col'].cumsum()                ║
║  PIVOT             → pd.pivot_table(...)                        ║
║  COUNT(*)          → pd.crosstab(col1, col2)                   ║
║                                                                   ║
║  agg()       → reduces (1 row per group)                       ║
║  transform() → same size (broadcasts back)                     ║
║  apply()     → anything (full flexibility)                      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | agg vs transform vs apply? | agg reduces to 1 row/group, transform keeps same size, apply is flexible |
| 2 | How to replicate SQL HAVING? | .groupby().filter(lambda g: condition) |
| 3 | pivot_table vs crosstab? | pivot_table = flexible agg with any function. crosstab = specialized for frequency counts |
| 4 | Add group-level stat to each row? | .groupby('col')['val'].transform('mean') — broadcasts back |
| 5 | Rank within groups? | df.groupby('dept')['salary'].rank(ascending=False) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • groupby = split-apply-combine                        │
## │  • Named agg for clean column names                     │
## │  • transform() broadcasts group stats back              │
## │  • filter() = SQL HAVING                                │
## │  • pivot_table for 2D aggregation summaries             │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Merge & Join** — merge(), join(), concat()